# 04 · Model router – by prompt complexity
Two switchable modes (`complexity-mode`):
* **heuristic** – policy scores code, reasoning verbs, multi-questions, math, conversation depth → tier. Free and fast.
* **llm** – a small classifier model labels the prompt `simple|medium|complex`; falls back to heuristic if it fails.

`router-mode = hybrid` takes the *higher* tier of length and complexity.

In [ ]:
import sys; sys.path.insert(0, '.')
import time
from datetime import datetime, timezone, timedelta
from heimdal_helpers import *

PROMPTS = {
  'trivia':   "What colour is the Space Stone?",
  'summary':  "Summarise the main arc of the Infinity Saga and compare Thor's and Iron Man's character development.",
  'complex':  "Design a step by step architecture for a multi-region event-driven platform. Analyze trade-offs, evaluate failure modes, "
              "and explain why you'd pick each component. Include a Python class for the retry logic. What are the pros and cons? How would you test it?",
}
USER = 'tony-stark'
RESULTS = []   # raw ask() results of the last run(), used by check_logs()

def run():
    """Send every prompt through the router and show the decision the gateway returned in its headers."""
    global RESULTS
    RESULTS = [{'prompt': k, 'text': v, **ask(v, user=USER)} for k, v in PROMPTS.items()]
    return show(RESULTS, cols=('tier', 'model', 'reason'))

def check_logs(results=None, timeout_min=12, interval_s=30):
    """Look the last run() up in the APIM LLM gateway log: which model really served which prompt?

    The log holds no prompt text, so each log row is matched to its prompt by user + prompt token count
    (the client gets the same count back in the response). Logs lag a few minutes, so this polls.
    """
    results = results or RESULTS
    want = {r['prompt_tokens']: r for r in results}
    q = f"""
    ApiManagementGatewayLlmLog
    | where TimeGenerated > ago(1h)
    | join kind=inner (ApiManagementGatewayLogs | project CorrelationId, User = ApimSubscriptionId) on CorrelationId
    | where User == '{USER}' and PromptTokens in ({', '.join(str(t) for t in want)})
    | project TimeGenerated, PromptTokens, CompletionTokens, LogModel = ModelName
    """
    deadline = time.time() + timeout_min * 60
    while True:
        logs = kql(q, hours=1).sort_values('TimeGenerated').drop_duplicates('PromptTokens', keep='last')
        if len(logs) >= len(want) or time.time() > deadline:
            break
        print(f"{len(logs)}/{len(want)} requests in the log yet - waiting {interval_s}s...")
        time.sleep(interval_s)
    df = pd.DataFrame(results).rename(columns={'model': 'header_model'})
    df = df.merge(logs.rename(columns={'PromptTokens': 'prompt_tokens'}), on='prompt_tokens', how='left')
    # the log reports model name + version (gpt-6.1-sol-2026-09-29); the header has the deployment name (gpt-6-1-sol)
    df['match'] = [str(l).replace('.', '-').startswith(h) for h, l in zip(df['header_model'], df['LogModel'])]
    return df[['prompt', 'tier', 'header_model', 'LogModel', 'match', 'prompt_tokens', 'CompletionTokens', 'reason']]

set_router(router_mode='complexity', complexity_mode='heuristic', complexity_threshold='3')
time.sleep(60)
run()

**Check the gateway log** – the log has no prompt text, so rows are matched to prompts by user + prompt token count. Takes a few minutes to appear:

In [ ]:
check_logs()   # which deployment actually served each prompt, according to the gateway log

Switch to the **LLM classifier** and compare decisions:

In [ ]:
set_router(complexity_mode='llm')
time.sleep(60)
run()

**Check the gateway log** (LLM gateway log, a few minutes of lag): was the classifier's tier really the model that served it? `llmTier` in the `reason` above is the classifier's verdict (0 simple, 1 medium, 2 complex); `-1` means the classifier call failed and the heuristic was used instead.

In [ ]:
check_logs()   # which deployment actually served each prompt, according to the gateway log

Back to the default **hybrid** (length + complexity, heuristic):

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

set_router(router_mode='hybrid', complexity_mode='heuristic', length_threshold_tokens='500')